In [ ]:
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl");
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/utils/FFT_perso.jl");

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 25.338776456203686
ω_b_ATS = 2*ω_a

φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv_ATS = Z_b*ω_b_ATS
E_Cb_ATS = C_b_inv_ATS/2

E_J = 37.12*2*pi*1.5


##Drive
ω_d=2ω_a# Drive frequency
ω_p=2ω_a-ω_d

ϵ_p = 0.58#0.62#
g2 = sin(ϵ_p)*E_J*φ_a^2*φ_b # static pump (ω_p = 0)
ϵ_d = 8.12*g2

buffer_phase = 0.0
pump_phase = 0.0;

##Env
κ_b= π/10.4; # 1/ns, same value and convention as the other notebooks (was 1/10.4)
κ_2 = 4*g2^2/κ_b
κ_a = 0#κ_b/2#3e-3*κ_2 #from Leon's paper

# First change of variables
Q_a, Q_b = 1/κ_a, 1/κ_b
Q_a_tilde, Q_b_tilde_ATS = ω_a/κ_a, ω_b_ATS/κ_b
β_a, β_b_ATS= 4E_J*sin(ϵ_p)φ_a^2/(ω_a), 4E_J*sin(ϵ_p)φ_b^2/ω_b_ATS
λ_a, λ_b = φ_a, φ_b
ϵ_λ_d_ATS = ϵ_d*2*λ_b/ω_b_ATS
ν_d = 1;

# Second change of variables
κ_a_tilde_ATS, κ_b_tilde_ATS = 1/(4*Q_a_tilde*sqrt(1-1/(4Q_b_tilde_ATS^2))), 1/(2*Q_b_tilde_ATS*sqrt(1-1/(4Q_b_tilde_ATS^2)))
β_a_tilde_ATS, β_b_tilde_ATS = β_a/(2-1/(2Q_b_tilde_ATS^2)), β_b_ATS/(1-1/(4Q_b_tilde_ATS^2))
ϵ_d_tilde_ATS = ϵ_λ_d_ATS/(1-1/(4Q_b_tilde_ATS^2))
ν_d_tilde_ATS = ν_d/sqrt(1-1/(4Q_b_tilde_ATS^2))
γ_ATS = (1-1/(4Q_a_tilde^2))/(1-1/(4Q_b_tilde_ATS^2))/2;

In [ ]:
t_max = 300/κ_b
t_min = 0.0
t_min_save = t_max - 2pi/(ω_a)*2
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save=2pi/(30sqrt(2)*3ω_a)
t_save = max(t_min_save,0):dt_save:t_max


b_0=0.0
environment = MeanField(κ_a, κ_b)


drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_p, ω_p, pump_phase)
drive_prepared = prepare(drive)

t_list = collect(range(max(t_min_save,0),step = dt_save,stop = t_max));

In [ ]:
Im_a_list=range(-6,6,10)
Re_a_list=range(-6,6,10)

Final_mode_a_ATS = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
Final_mode_b_ATS = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
Final_mode_a_ATS_cat = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
Final_mode_b_ATS_cat = Dict{Tuple{Float64,Float64}, Vector{ComplexF32}}()
for i in 1:length(Re_a_list)
    re_i = Re_a_list[i]
    for j in 1:length(Im_a_list)
        im_j = Im_a_list[j]
        a_0_j = re_i + 1im*im_j

        system = SystemATS_ladder(ω_a, ω_b_ATS, E_J, E_Ca, E_Cb_ATS, a_0_j, b_0)
        system_prepared = prepare(system)
        environment_prepared = prepare(system,environment);

        u_0 = get_init(system_prepared, environment_prepared)
        p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

        a, b, t_list = default_solver(u_0, (t_min, t_max), p, dt, t_save = t_save, equation = default_equation!)
        x_a, p_a = (a+conj(a))/sqrt(2), 1im*(conj(a)-a)/sqrt(2)
        x_b, p_b = (b+conj(b))/sqrt(2), 1im*(conj(b)-b)/sqrt(2)

        x_lam_a, x_lam_b, p_lam_a, p_lam_b = sqrt(2)*x_a, sqrt(2)*x_b, sqrt(2)*p_a, sqrt(2)*p_b
        u_a =@. cos(ω_b_ATS*t_list/2)*x_lam_a - sin(ω_b_ATS*t_list/2)*p_lam_a
        v_a =@. sin(ω_b_ATS*t_list/2)*x_lam_a + cos(ω_b_ATS*t_list/2)*p_lam_a
        u_b =@. cos(ω_b_ATS*t_list)*x_lam_b- sin(ω_b_ATS*t_list)*p_lam_b
        v_b =@. sin(ω_b_ATS*t_list)*x_lam_b + cos(ω_b_ATS*t_list)*p_lam_b;

        a_R = (u_a+1im*v_a)/2*1im # = i a e^{iω_a t}, in units of a (u+iv = 2a e^{iωt}; was /sqrt(2), i.e. √2 a, until 12/10/2026)
        b_R = (u_b+1im*v_b)/2*1im #Rotation 90°
        if maximum(abs.(imag(a_R))) > 1.0   # satellites sit at |Im a| ≈ 4.2, cats on the real axis (was > 0.1, too tight for the larger cats)
            Final_mode_a_ATS[(re_i,im_j)] = a_R
            Final_mode_b_ATS[(re_i,im_j)] = b_R
        else 
            Final_mode_a_ATS_cat[(re_i,im_j)] = a_R
            Final_mode_b_ATS_cat[(re_i,im_j)] = b_R
        end    
    end
end

In [ ]:
#prediction
b_0_norm = 1/(12*(β_a/2+β_b_ATS/2))+sqrt((1/(12*(β_a/2+β_b_ATS/2)))^2+ϵ_λ_d_ATS/3/β_b_ATS)#0.40239616164970887
#a_0_norm = 0.4920305631589086

gamma = 1 + β_b_ATS/β_a
a_1 = -b_0_norm/(2*gamma)#- β_a / 2*(0.4920305631589086^2+0.40239616164970887^2)
a_0_norm = sqrt(-a_1/(β_a/2)-b_0_norm^2)

b_m1_norm = 2β_b_ATS /2*b_0_norm*a_0_norm
b_2 = β_b_ATS * a_1 / β_a

θ_b_0 = -2*b_0_norm/(ϵ_λ_d_ATS/2)*κ_b/ω_b_ATS#-0.11041398179241003
θ_a_0_1 = θ_b_0/2
θ_a_0_2 = θ_b_0/2+π
θ_b_m1 = 3*θ_a_0_1

b_0 = b_0_norm*exp(1im*θ_b_0)
a_0_1 = a_0_norm*exp(1im*θ_a_0_1)
a_0_2 = a_0_norm*exp(1im*θ_a_0_2)
b_m1 = b_m1_norm*exp(1im*θ_b_m1)

b_0_norm_prec = 0.33636495810132455
a_0_norm_prec = 0.47287700802978716

a_1_prec = -0.06895043097048979
b_m1_norm_prec = -0.07201944643601312
b_2_prec = β_b_ATS * a_1_prec / β_a

θ_b_0_prec = -0.141756683610786
θ_a_0_1_prec = θ_b_0_prec/2
θ_a_0_2_prec = θ_b_0_prec/2+π
θ_b_m1_prec = 3*θ_a_0_1_prec

b_0_prec = b_0_norm_prec*exp(1im*θ_b_0_prec)
a_0_1_prec = a_0_norm_prec*exp(1im*θ_a_0_1_prec)
a_0_2_prec = a_0_norm_prec*exp(1im*θ_a_0_2_prec)

b_0_cat_state = 0
a_0_cat_state_1 = 1im*sqrt(ϵ_d*λ_b/ω_b_ATS/(β_b_ATS/2))
a_0_cat_state_2 = -1im*sqrt(ϵ_d*λ_b/ω_b_ATS/(β_b_ATS/2));


In [ ]:
using JSON
satellite_minus_position_dict = JSON.parsefile("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/satellites_minus_12_10_2026_Hill.json")
satellite_plus_position_dict = JSON.parsefile("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/satellites_plus_12_10_2026_Hill.json")
satellite_buffer_position_dict = JSON.parsefile("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/analysis/output/satellites_buffer_12_10_2026_Hill.json")
Re_a_list_satellite_minus = []
Im_a_list_satellite_minus = []
for entry in satellite_minus_position_dict
    push!(Re_a_list_satellite_minus,-entry["imag"]) #Everything is rotated 90°
    push!(Im_a_list_satellite_minus,entry["real"])
end
Re_a_list_satellite_minus /= λ_a   # Hill a_R = (x+ip)/√2 with x + ip = √2 φ_a a (was /(λ_a/√2), i.e. √2 a)
Im_a_list_satellite_minus /= λ_a

Re_a_list_satellite_plus = []
Im_a_list_satellite_plus = []
for entry in satellite_plus_position_dict
    push!(Re_a_list_satellite_plus,-entry["imag"])
    push!(Im_a_list_satellite_plus,entry["real"])
end
Re_a_list_satellite_plus /= λ_a
Im_a_list_satellite_plus /= λ_a

Re_b_list_satellite = []
Im_b_list_satellite = []
for entry in satellite_buffer_position_dict
    push!(Re_b_list_satellite,-entry["imag"])
    push!(Im_b_list_satellite,entry["real"])
end
Re_b_list_satellite /= λ_b
Im_b_list_satellite /= λ_b

In [ ]:
p0 = plot()
for key in keys(Final_mode_a_ATS)
    z = Final_mode_a_ATS[key]
    scatter!(p0, real.(z), imag.(z), ms = 1, color = RGBA(185/256,21/256,21/256,1), label = "",)
end
for key in keys(Final_mode_a_ATS_cat)
    z = Final_mode_a_ATS_cat[key]
    scatter!(p0, real.(z), imag.(z), ms = 1, color = RGBA(13/256,76/256,116/256,1), label = "",)
end

# Predictions
angle_shift = 0.08
plot!(p0, Re_a_list_satellite_minus .*cos(angle_shift/2) .+ Im_a_list_satellite_minus .*sin(angle_shift/2), Im_a_list_satellite_minus .*cos(angle_shift/2) .-  Re_a_list_satellite_minus.*sin(angle_shift/2), label="", color=RGBA(242/256,199/256,67/256,1), ls=:dash)
plot!(p0, Re_a_list_satellite_plus .*cos(angle_shift/2) .+ Im_a_list_satellite_plus .*sin(angle_shift/2), Im_a_list_satellite_plus .*cos(angle_shift/2) .-  Re_a_list_satellite_plus.*sin(angle_shift/2), label= "", color=RGBA(242/256,199/256,67/256,1), ls=:dash)
xlims!(p0, (-3.5,3.5))
ylims!(p0, (-5.7,5.7))
plot!(p0, xlabel = L"\mathrm{Re}(a)", ylabel = L"\mathrm{Im}(a)", aspect_ratio = 1, guidefontsize=12)
#plot!(p0, real(a_R), imag(a_R), label="", color = RGBA(13/256,76/256,116/256,0.2))

p1 = plot()
for key in keys(Final_mode_b_ATS)
    z = Final_mode_b_ATS[key]
    scatter!(p1, real.(z), imag.(z), ms = 1, color = RGBA(185/256,21/256,21/256,1), label = "",)
end
for key in keys(Final_mode_b_ATS_cat)
    z = Final_mode_b_ATS_cat[key]
    scatter!(p1, real.(z), imag.(z), ms = 1, color = RGBA(13/256,76/256,116/256,1), label = "",)
end
plot!(p1, Re_b_list_satellite .*cos(angle_shift) .+ Im_b_list_satellite .*sin(angle_shift), Im_b_list_satellite .*cos(angle_shift) .-  Re_b_list_satellite.*sin(angle_shift), label="", color=RGBA(242/256,199/256,67/256,1), ls=:dash)
xlims!(p1, (-0.7,1.6))
ylims!(p1, (-0.35,2.85))
plot!(p1, xlabel = L"\mathrm{Re}(b)", ylabel = L"\mathrm{Im}(b)", aspect_ratio = 1, guidefontsize=12)
#plot!(p1, real(b_R), imag(b_R), label="", color = RGBA(13/256,76/256,116/256,0.2))
plot(p0, p1, layout = (1, 2))
#savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/poster_chicago/classical_limit_cycles.pdf")

In [ ]:
p0 = plot()
for key in keys(Final_mode_a_ATS)
    z = Final_mode_a_ATS[key]
    scatter!(p0, real.(z), imag.(z), ms = 1, color = RGBA(185/256,21/256,21/256,1), label = "",)
end
for key in keys(Final_mode_a_ATS_cat)
    z = Final_mode_a_ATS_cat[key]
    scatter!(p0, real.(z), imag.(z), ms = 1, color = RGBA(13/256,76/256,116/256,1), label = "",)
end

# Predictions
angle_shift = 0.08
#plot!(p0, Re_a_list_satellite_minus .*cos(angle_shift/2) .+ Im_a_list_satellite_minus .*sin(angle_shift/2), Im_a_list_satellite_minus .*cos(angle_shift/2) .-  Re_a_list_satellite_minus.*sin(angle_shift/2), label="", color=RGBA(242/256,199/256,67/256,1), ls=:dash)
#plot!(p0, Re_a_list_satellite_plus .*cos(angle_shift/2) .+ Im_a_list_satellite_plus .*sin(angle_shift/2), Im_a_list_satellite_plus .*cos(angle_shift/2) .-  Re_a_list_satellite_plus.*sin(angle_shift/2), label= "", color=RGBA(242/256,199/256,67/256,1), ls=:dash)
xlims!(p0, (-3.5,3.5))
ylims!(p0, (-5.7,5.7))
plot!(p0, xlabel = L"\mathrm{Re}(a)", ylabel = L"\mathrm{Im}(a)", aspect_ratio = 1, guidefontsize=12)
#plot!(p0, real(a_R), imag(a_R), label="", color = RGBA(13/256,76/256,116/256,0.2))

p1 = plot()
for key in keys(Final_mode_b_ATS)
    z = Final_mode_b_ATS[key]
    scatter!(p1, real.(z), imag.(z), ms = 1, color = RGBA(185/256,21/256,21/256,1), label = "",)
end
for key in keys(Final_mode_b_ATS_cat)
    z = Final_mode_b_ATS_cat[key]
    scatter!(p1, real.(z), imag.(z), ms = 1, color = RGBA(13/256,76/256,116/256,1), label = "",)
end
#plot!(p1, Re_b_list_satellite .*cos(angle_shift) .+ Im_b_list_satellite .*sin(angle_shift), Im_b_list_satellite .*cos(angle_shift) .-  Re_b_list_satellite.*sin(angle_shift), label="", color=RGBA(242/256,199/256,67/256,1), ls=:dash)
xlims!(p1, (-0.7,1.6))
ylims!(p1, (-0.35,2.85))
plot!(p1, xlabel = L"\mathrm{Re}(b)", ylabel = L"\mathrm{Im}(b)", aspect_ratio = 1, guidefontsize=12)
#plot!(p1, real(b_R), imag(b_R), label="", color = RGBA(13/256,76/256,116/256,0.2))
plot(p0, p1, layout = (1, 2))
#savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/poster_chicago/classical_limit_cycles_dynamics_only.pdf")

In [ ]:
p0 = plot()

for key in keys(Final_mode_a_ATS_cat)
    z = Final_mode_a_ATS_cat[key]
    scatter!(p0, real.(z), imag.(z), ms = 1, color = RGBA(13/256,76/256,116/256,1), label = "",)
end

# Predictions
angle_shift = 0.08


xlims!(p0, (-3.5,3.5))
ylims!(p0, (-5.7,5.7))
plot!(p0, xlabel = L"\mathrm{Re}(a)", ylabel = L"\mathrm{Im}(a)", aspect_ratio = 1, guidefontsize=12)
#plot!(p0, real(a_R), imag(a_R), label="", color = RGBA(13/256,76/256,116/256,0.2))

p1 = plot()

for key in keys(Final_mode_b_ATS_cat)
    z = Final_mode_b_ATS_cat[key]
    scatter!(p1, real.(z), imag.(z), ms = 1, color = RGBA(13/256,76/256,116/256,1), label = "",)
end

xlims!(p1, (-0.7,1.6))
ylims!(p1, (-0.35,2.85))
plot!(p1, xlabel = L"\mathrm{Re}(b)", ylabel = L"\mathrm{Im}(b)", aspect_ratio = 1, guidefontsize=12)
#plot!(p1, real(b_R), imag(b_R), label="", color = RGBA(13/256,76/256,116/256,0.2))
plot(p0, p1, layout = (1, 2))
#savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/poster_chicago/classical_limit_cycles_cat_only.pdf")

In [ ]:
t_max = 8/κ_b
t_min = 0.0
tspan = (t_min, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save=2pi/(ω_a)
t_save = max(t_min,0):dt_save:t_max


environment = MeanField(κ_a, κ_b)

drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_p, ω_p, pump_phase)
drive_prepared = prepare(drive)

t_list = collect(range(max(t_min_save,0),step = dt_save,stop = t_max));

re_i = 1/sqrt(2)#-.5/sqrt(2)
im_j = -.5/sqrt(2)#.5/sqrt(2)
a_0 = re_i + 1im*im_j
b_0 = 3/sqrt(2) + 1im*0/sqrt(2)

system = SystemATS_ladder(ω_a, ω_b_ATS, E_J, E_Ca, E_Cb_ATS, a_0, b_0)
system_prepared = prepare(system)
environment_prepared = prepare(system,environment);

u_0 = get_init(system_prepared, environment_prepared)
p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

a, b, t_list = default_solver(u_0, (t_min, t_max), p, dt, t_save = t_save, equation = default_equation!)
x_a, p_a = (a+conj(a))/sqrt(2), 1im*(conj(a)-a)/sqrt(2)
x_b, p_b = (b+conj(b))/sqrt(2), 1im*(conj(b)-b)/sqrt(2)

x_lam_a, x_lam_b, p_lam_a, p_lam_b = sqrt(2)*x_a, sqrt(2)*x_b, sqrt(2)*p_a, sqrt(2)*p_b
u_a =@. cos(ω_b_ATS*t_list/2)*x_lam_a - sin(ω_b_ATS*t_list/2)*p_lam_a
v_a =@. sin(ω_b_ATS*t_list/2)*x_lam_a + cos(ω_b_ATS*t_list/2)*p_lam_a
u_b =@. cos(ω_b_ATS*t_list)*x_lam_b- sin(ω_b_ATS*t_list)*p_lam_b
v_b =@. sin(ω_b_ATS*t_list)*x_lam_b + cos(ω_b_ATS*t_list)*p_lam_b;

a_R = (u_a+1im*v_a)/2*1im # = i a e^{iω_a t}, in units of a (u+iv = 2a e^{iωt}; was /sqrt(2), i.e. √2 a, until 12/10/2026)
b_R = (u_b+1im*v_b)/2*1im; #Rotation 90°

In [ ]:
plot(real(a_R), imag(a_R))

In [ ]:
plot(real(b_R), imag(b_R))

# Satellites: analytic harmonic balance

Implements the appendix "Satellites", with the primed parameters of the Hill notebooks:
$\beta_a' = 2E_J\sin\varepsilon_p\,\varphi_a^2/\omega_a$, $\beta_b' = 2E_J\sin\varepsilon_p\,\varphi_b^2/\omega_b$,
$\varepsilon_d' = \varepsilon_d\varphi_b/\omega_b$, $\kappa_b' = \kappa_b/\omega_b$.

- **Full system:** the scalar equation for $\bar y$, with $u(\bar y)$, $\varepsilon_\text{eff}(y)$ and the dissipative tilt $2\theta = -\arcsin(2\kappa_b' y/\varepsilon_d')$.
- **Limit $u\to0$, $\kappa_b'\to0$:** the closed-form $\bar y$, $|a_0|$ and $|b_0|$, with $\theta = 0$.

Orbits $a(\tau) = a_0 + a_1e^{i\tau} + a_{-1}e^{-i\tau}$ and $b(\tau) = b_0 + b_{-1}e^{-i\tau} + b_2e^{2i\tau}$ are drawn in the frame of the
trajectories above: $a_R = i\,a(\tau)/\varphi_a$ and $b_R = i\,b(\tau)/\varphi_b$.

In [ ]:
# Primed parameters (same definitions as in the Hill notebooks)
β_a′ = 2E_J*sin(ϵ_p)*φ_a^2/ω_a
β_b′ = 2E_J*sin(ϵ_p)*φ_b^2/ω_b_ATS
ϵ_d′ = ϵ_d*φ_b/ω_b_ATS
κ_b′ = κ_b/ω_b_ATS
β_Σ = β_a′ + β_b′
γ_sat = 1 + β_b′/(β_a′*(1 + κ_b′^2))

ϵ_eff(y) = sqrt(max(ϵ_d′^2 - 4κ_b′^2*y^2, 0.0))

# y from ȳ: y = ȳ + β_a′ ϵ_eff(y)/(2β_b′), solved by fixed-point iteration (κ_b′ ≪ 1)
function y_of_ȳ(ȳ)
    y = ȳ + β_a′*ϵ_d′/(2β_b′)
    for _ in 1:100
        y = ȳ + β_a′*ϵ_eff(y)/(2β_b′)
    end
    return y
end

# root of η u² + (1+η) u - η = 0 continuously connected to u = 0
function u_of_ȳ(ȳ)
    η = 2β_b′*ȳ
    return (sqrt((1 + η)^2 + 4η^2) - (1 + η))/(2η)
end

function x²_of_ȳ(ȳ)
    u = u_of_ȳ(ȳ)
    return (2(1 + u)*ȳ^2 - ϵ_eff(y_of_ȳ(ȳ))/β_b′)/(1 + 2u)
end

# scalar equation for ȳ
function scalar_residual(ȳ)
    u = u_of_ȳ(ȳ)
    return ȳ*(1 + u) - 2β_Σ*((1 + (1 + u)^2/2)*ȳ^2 + (1 + u^2)*x²_of_ȳ(ȳ))
end

# all roots on a grid (bisection inside each sign change); the physical ones have ȳ > 0 and x² > 0
function roots_on_grid(f, grid)
    roots = Float64[]
    for k in 1:length(grid)-1
        lo, hi = grid[k], grid[k+1]
        f(lo)*f(hi) > 0 && continue
        for _ in 1:200
            mid = (lo + hi)/2
            f(lo)*f(mid) <= 0 ? (hi = mid) : (lo = mid)
        end
        push!(roots, (lo + hi)/2)
    end
    return roots
end

ȳ_roots = roots_on_grid(scalar_residual, range(1e-4, 3.0, length=30_000))
ȳ_physical = filter(ȳ -> x²_of_ȳ(ȳ) > 0, ȳ_roots)
println("roots ȳ: ", ȳ_roots, "   physical (x² > 0): ", ȳ_physical)

# amplitudes of one satellite (s = ±1 gives the pair related by a → -a)
function satellite(ȳ, x, u, y, ϵ_e, γ, θ, s)
    a_1 = -(1 + u)*ȳ/(2γ)
    return (a_0 = s*x*cis(θ), a_1 = a_1, a_m1 = -β_a′*ϵ_e/(2β_b′)*cis(2θ),
            b_0 = y*cis(2θ), b_m1 = s*u*x*cis(3θ), b_2 = β_b′/β_a′*a_1)
end

# full system (u ≠ 0, κ_b′ kept in ϵ_eff, γ and the tilt)
ȳ_full = ȳ_physical[1]
u_full, y_full = u_of_ȳ(ȳ_full), y_of_ȳ(ȳ_full)
x_full = sqrt(x²_of_ȳ(ȳ_full))
θ_full = -asin(2κ_b′*y_full/ϵ_d′)/2
sat_full = [satellite(ȳ_full, x_full, u_full, y_full, ϵ_eff(y_full), γ_sat, θ_full, s) for s in (1, -1)]

# limit u → 0, κ_b′ → 0 (closed form, θ = 0, γ = β_Σ/β_a′)
ȳ_0 = (1 + sqrt(1 + 56β_Σ^2*ϵ_d′/β_b′))/(14β_Σ)
x_0 = sqrt(2ȳ_0^2 - ϵ_d′/β_b′)
y_0 = ȳ_0 + β_a′*ϵ_d′/(2β_b′)
sat_0 = [satellite(ȳ_0, x_0, 0.0, y_0, ϵ_d′, β_Σ/β_a′, 0.0, s) for s in (1, -1)]

# orbits over one memory period, in the frame of the trajectories
τ_orbit = range(0, 2π, length=400)
orbit_a(S) = @. 1im*(S.a_0 + S.a_1*cis(τ_orbit) + S.a_m1*cis(-τ_orbit))/φ_a
orbit_b(S) = @. 1im*(S.b_0 + S.b_m1*cis(-τ_orbit) + S.b_2*cis(2τ_orbit))/φ_b

println("full system : ȳ = ", ȳ_full, ", u = ", u_full, ", |a_0| = ", x_full, ", |b_0| = ", y_full, ", θ = ", θ_full)
println("u → 0 limit : ȳ = ", ȳ_0, ", |a_0| = ", x_0, ", |b_0| = ", y_0)
println("in plot units, |a_0|/φ_a: full = ", x_full/φ_a, ", u → 0 = ", x_0/φ_a,
        "   |b_0|/φ_b: full = ", y_full/φ_b, ", u → 0 = ", y_0/φ_b)

In [ ]:
# Trajectories (red: satellites/other, blue: cats), Hill prediction (yellow), analytic harmonic balance
color_full = RGBA(2/256, 147/256, 97/256, 1)   # full system (u ≠ 0)
color_u0 = RGBA(253/256, 108/256, 158/256, 1)            # limit u → 0
color_hill = RGBA(242/256, 199/256, 67/256, 1)

p0 = plot()
for key in keys(Final_mode_a_ATS)
    z = Final_mode_a_ATS[key]
    scatter!(p0, real.(z), imag.(z), ms = 1, color = RGBA(185/256,21/256,21/256,1), label = "",)
end
for key in keys(Final_mode_a_ATS_cat)
    z = Final_mode_a_ATS_cat[key]
    scatter!(p0, real.(z), imag.(z), ms = 1, color = RGBA(13/256,76/256,116/256,1), label = "",)
end
angle_shift = 0.0   # was 0.08 with κ_b = 1/10.4 here and an older Hill orbit; same κ_b and conventions now, no rotation needed
angle_shift_HB = angle_shift   # same rotation as the Hill curves (set to 0 for the bare HB prediction)
rotate(z, α) = z .* cis(-α)    # same convention as the Hill curves: (Re cos α + Im sin α, Im cos α - Re sin α)
plot!(p0, Re_a_list_satellite_minus .*cos(angle_shift/2) .+ Im_a_list_satellite_minus .*sin(angle_shift/2), Im_a_list_satellite_minus .*cos(angle_shift/2) .-  Re_a_list_satellite_minus.*sin(angle_shift/2), label="", color=color_hill, ls=:dash)
plot!(p0, Re_a_list_satellite_plus .*cos(angle_shift/2) .+ Im_a_list_satellite_plus .*sin(angle_shift/2), Im_a_list_satellite_plus .*cos(angle_shift/2) .-  Re_a_list_satellite_plus.*sin(angle_shift/2), label="", color=color_hill, ls=:dash)
for (k, S) in enumerate(sat_full)
    z = rotate(orbit_a(S), angle_shift_HB/2)
    plot!(p0, real.(z), imag.(z), color = color_full, ls=:dot, lw = 2, label="")
end
for (k, S) in enumerate(sat_0)
    z = rotate(orbit_a(S), angle_shift_HB/2)
    plot!(p0, real.(z), imag.(z), color = color_u0, lw = 2, ls=:dot, label="")
end
xlims!(p0, (-3.5,3.5))
ylims!(p0, (-5.7,5.7))
plot!(p0, xlabel = L"\mathrm{Re}(a)", ylabel = L"\mathrm{Im}(a)", aspect_ratio = 1, guidefontsize=12)

p1 = plot()
for key in keys(Final_mode_b_ATS)
    z = Final_mode_b_ATS[key]
    scatter!(p1, real.(z), imag.(z), ms = 1, color = RGBA(185/256,21/256,21/256,1), label = "",)
end
for key in keys(Final_mode_b_ATS_cat)
    z = Final_mode_b_ATS_cat[key]
    scatter!(p1, real.(z), imag.(z), ms = 1, color = RGBA(13/256,76/256,116/256,1), label = "",)
end
plot!(p1, Re_b_list_satellite .*cos(angle_shift) .+ Im_b_list_satellite .*sin(angle_shift), Im_b_list_satellite .*cos(angle_shift) .-  Re_b_list_satellite.*sin(angle_shift), label="", color=color_hill, ls=:dash)
for S in sat_full
    z = rotate(orbit_b(S), angle_shift_HB)
    plot!(p1, real.(z), imag.(z), color = color_full, lw = 2, label = "", ls=:dot)
end
for S in sat_0
    z = rotate(orbit_b(S), angle_shift_HB)
    plot!(p1, real.(z), imag.(z), color = color_u0, lw = 2, ls = :dot, label = "")
end
xlims!(p1, (-0.7,1.6))
ylims!(p1, (-0.35,2.85))
plot!(p1, xlabel = L"\mathrm{Re}(b)", ylabel = L"\mathrm{Im}(b)", aspect_ratio = 1, guidefontsize=12)
plot(p0, p1, layout = (1, 2))
savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/article/classical_limit_cycles_analytic_HB_12_10_2026.pdf")

In [ ]:
# Fig. 3 of the article: the figure above without the analytic harmonic-balance orbits (green, pink),
# i.e. trajectories (red: satellites, blue: cats) and the Hill orbit (yellow)
color_hill = RGBA(242/256, 199/256, 67/256, 1)
angle_shift = 0.0

p0 = plot()
for key in keys(Final_mode_a_ATS)
    z = Final_mode_a_ATS[key]
    scatter!(p0, real.(z), imag.(z), ms = 1, color = RGBA(185/256,21/256,21/256,1), label = "",)
end
for key in keys(Final_mode_a_ATS_cat)
    z = Final_mode_a_ATS_cat[key]
    scatter!(p0, real.(z), imag.(z), ms = 1, color = RGBA(13/256,76/256,116/256,1), label = "",)
end
plot!(p0, Re_a_list_satellite_minus .*cos(angle_shift/2) .+ Im_a_list_satellite_minus .*sin(angle_shift/2), Im_a_list_satellite_minus .*cos(angle_shift/2) .-  Re_a_list_satellite_minus.*sin(angle_shift/2), label="", color=color_hill, ls=:dash)
plot!(p0, Re_a_list_satellite_plus .*cos(angle_shift/2) .+ Im_a_list_satellite_plus .*sin(angle_shift/2), Im_a_list_satellite_plus .*cos(angle_shift/2) .-  Re_a_list_satellite_plus.*sin(angle_shift/2), label="", color=color_hill, ls=:dash)
xlims!(p0, (-3.5,3.5))
ylims!(p0, (-5.7,5.7))
plot!(p0, xlabel = L"\mathrm{Re}(a)", ylabel = L"\mathrm{Im}(a)", aspect_ratio = 1, guidefontsize=12)

p1 = plot()
for key in keys(Final_mode_b_ATS)
    z = Final_mode_b_ATS[key]
    scatter!(p1, real.(z), imag.(z), ms = 1, color = RGBA(185/256,21/256,21/256,1), label = "",)
end
for key in keys(Final_mode_b_ATS_cat)
    z = Final_mode_b_ATS_cat[key]
    scatter!(p1, real.(z), imag.(z), ms = 1, color = RGBA(13/256,76/256,116/256,1), label = "",)
end
plot!(p1, Re_b_list_satellite .*cos(angle_shift) .+ Im_b_list_satellite .*sin(angle_shift), Im_b_list_satellite .*cos(angle_shift) .-  Re_b_list_satellite.*sin(angle_shift), label="", color=color_hill, ls=:dash)
xlims!(p1, (-0.7,1.6))
ylims!(p1, (-0.35,2.85))
plot!(p1, xlabel = L"\mathrm{Re}(b)", ylabel = L"\mathrm{Im}(b)", aspect_ratio = 1, guidefontsize=12)
plot(p0, p1, layout = (1, 2))
savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/article/classical_limit_cycles_HB_12_10_2026.pdf")